# Weryfikacja SkateFormer na NTU-60 analiza wpływu preprocessingu
Etap I test wpływu preprocessingu na wyniki klasyfikacji

sprawdzenie wpływu interpolacji i normalizacji skali na wyniki SkateFormer na oryginalnych danych NTU-60. Wcześniejsze eksperymenty na zbiorze PKU-MMD wskazywały na poprawę wyniku po zastosowaniu tych operacji, dlatego sprawdzono ich wpływ również na NTU-60.

| Plik danych | Preprocessing | Wagi |
|---|---|---|
| NTU60_CS.npz | baseline | ntu60_CSub |
| NTU60_CS_scale.npz | scale only | ntu60_CSub |
| NTU60_CS_both.npz | interpolacja + scale | ntu60_CSub |
| NTU60_CV.npz | baseline | ntu60_CSub |
| NTU60_CV_scale.npz | scale only | ntu60_CSub |
| NTU60_CV_both.npz | interpolacja + scale | ntu60_CSub |

Punkt odniesienia: dla wariantu bazowego oczekiwany jest wynik około 92,9% dla CS. Wyniki poszczególnych wariantów preprocessingu zostaną porównane z wariantem bazowym.

In [ ]:
!git clone https://github.com/KAIST-VICLab/SkateFormer.git
import os
os.chdir('/content/SkateFormer')
!ls

Cloning into 'SkateFormer'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 180 (delta 52), reused 43 (delta 35), pack-reused 101 (from 1)
Receiving objects: 100% (180/180), 1.12 MiB | 27.21 MiB/s, done.
Resolving deltas: 100% (80/80), done.
assets	data	 LICENSE  model		  README.md	     skateformer
config	feeders  main.py  pyproject.toml  requirements.yaml  torchlight


In [ ]:
!pip install -q einops==0.6.1 timm==0.9.12 tensorpack torchpack loguru msgpack msgpack-numpy tabulate tensorboardX

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.3/296.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil, numpy as np, os

DRIVE_ROOT    = '/content/drive/MyDrive'
DRIVE_WEIGHTS_CS = f'{DRIVE_ROOT}/SkateFormer_weights/ntu60_CSub'
DRIVE_WEIGHTS_CV = f'{DRIVE_ROOT}/SkateFormer_weights/ntu60_CView'

os.makedirs('/content/SkateFormer/data/ntu', exist_ok=True)

NTU_FILES = [
    'NTU60_CS.npz',
    'NTU60_CV.npz',
    'NTU60_CS_scale.npz',
    'NTU60_CV_scale.npz',
    'NTU60_CS_both.npz',
    'NTU60_CV_both.npz',
    'NTU60_CS_interpolation.npz',
    'NTU60_CV_interpolation.npz',
    'NTU60_CS_scale_per_actor.npz',
    'NTU60_CV_scale_per_actor.npz',
    'NTU60_CS_both_per_actor.npz',
    'NTU60_CV_both_per_actor.npz',
]

EXPECTED_TEST_N = {'CS': 16487, 'CV': 18932}

print(f'{"plik":38s} {"train":>10s} {"test":>10s} {"status"}')
for fname in NTU_FILES:
    src = f'{DRIVE_ROOT}/{fname}'
    dst = f'/content/SkateFormer/data/ntu/{fname}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)

    d = np.load(dst)
    x_train_shape = d['x_train'].shape
    x_test_shape  = d['x_test'].shape

    split = 'CS' if fname.startswith('NTU60_CS') else 'CV'
    expected_n = EXPECTED_TEST_N[split]

    ok = (len(x_test_shape) == 3 and x_test_shape[2] == 150
          and x_test_shape[0] == expected_n)
    status = 'OK' if ok else '!! PODEJRZANY KSZTAŁT'

    print(f'{fname:38s} {str(x_train_shape):>10s} {str(x_test_shape):>10s} {status}')

pt_src = f'{DRIVE_WEIGHTS_CS}/SkateFormer_j.pt'
pt_dst = '/content/SkateFormer/SkateFormer_j_CSub.pt'
if not os.path.exists(pt_dst):
    shutil.copy(pt_src, pt_dst)
print(f'Skopiowano: SkateFormer_j_CSub.pt  ({os.path.getsize(pt_dst)/1024**2:.1f} MB)')

pt_src = f'{DRIVE_WEIGHTS_CV}/SkateFormer_j.pt'
pt_dst = '/content/SkateFormer/SkateFormer_j_CView.pt'
if not os.path.exists(pt_dst):
    shutil.copy(pt_src, pt_dst)
print(f'Skopiowano: SkateFormer_j_CView.pt  ({os.path.getsize(pt_dst)/1024**2:.1f} MB)')

plik                                        train       test status
NTU60_CS.npz                           (40091, 300, 150) (16487, 300, 150) OK
NTU60_CV.npz                           (37646, 300, 150) (18932, 300, 150) OK
NTU60_CS_scale.npz                     (40091, 300, 150) (16487, 300, 150) OK
NTU60_CV_scale.npz                     (37646, 300, 150) (18932, 300, 150) OK
NTU60_CS_both.npz                      (40091, 300, 150) (16487, 300, 150) OK
NTU60_CV_both.npz                      (37646, 300, 150) (18932, 300, 150) OK
NTU60_CS_interpolation.npz             (40091, 300, 150) (16487, 300, 150) OK
NTU60_CV_interpolation.npz             (37646, 300, 150) (18932, 300, 150) OK
NTU60_CS_scale_per_actor.npz           (40091, 300, 150) (16487, 300, 150) OK
NTU60_CV_scale_per_actor.npz           (37646, 300, 150) (18932, 300, 150) OK
NTU60_CS_both_per_actor.npz            (40091, 300, 150) (16487, 300, 150) OK
NTU60_CV_both_per_actor.npz            (37646, 300, 150) (18932, 300, 150)

In [ ]:
import os, sys
os.chdir('/content/SkateFormer')

#Poprawka torchlight
with open('/content/SkateFormer/torchlight/torchlight/__init__.py', 'w') as f:
    f.write('''import argparse

class DictAction(argparse.Action):
    def __call__(self, parser, namespace, values, option_string=None):
        input_dict = getattr(namespace, self.dest, {}) or {}
        for kv in values:
            key, val = kv.split("=", 1)
            try:
                val = eval(val)
            except:
                pass
            input_dict[key] = val
        setattr(namespace, self.dest, input_dict)

class IO:
    pass
''')
print('Zastosowano poprawkę torchlight.')

#Usunięcie konfliktu z zainstalowaną wersją torchlight
os.system('pip uninstall torchlight -y')
if 'torchlight' in sys.modules:
    del sys.modules['torchlight']
sys.path.insert(0, '/content/SkateFormer/torchlight')
print('Usunięto konflikt z systemowym torchlight.')

#Poprawka main.py
with open('/content/SkateFormer/main.py', 'r') as f:
    content = f.read()
content = content.replace(
    'from torchlight import DictAction',
    '''class DictAction(argparse.Action):
    def __call__(self, parser, namespace, values, option_string=None):
        input_dict = getattr(namespace, self.dest, {}) or {}
        for kv in values:
            key, val = kv.split("=", 1)
            try:
                val = eval(val)
            except:
                pass
            input_dict[key] = val
        setattr(namespace, self.dest, input_dict)'''
)
content = content.replace('default_arg = yaml.load(f)',
                          'default_arg = yaml.load(f, Loader=yaml.SafeLoader)')
with open('/content/SkateFormer/main.py', 'w') as f:
    f.write(content)
print('Zastosowano poprawkę main.py.')

#Zastąpienie przestarzałego np.int
with open('/content/SkateFormer/feeders/feeder_ntu.py', 'r') as f:
    content = f.read()
content = content.replace('.astype(np.int)', '.astype(int)')
with open('/content/SkateFormer/feeders/feeder_ntu.py', 'w') as f:
    f.write(content)
print('Zastąpiono przestarzały typ np.int.')

#Zastąpienie przestarzałych typów NumPy
with open('/content/SkateFormer/feeders/tools.py', 'r') as f:
    content = f.read()
for old, new in [('np.int)', 'int)'), ('np.int,', 'int,'),
                 ('np.float)', 'float)'), ('np.float,', 'float,'),
                 ('np.bool)', 'bool)'), ('np.complex)', 'complex)')]:
    content = content.replace(old, new)
with open('/content/SkateFormer/feeders/tools.py', 'w') as f:
    f.write(content)
print('Zastąpiono przestarzałe typy NumPy.')

print('\nZakończono dostosowanie kodu SkateFormer.')

Zastosowano poprawkę torchlight.
Usunięto konflikt z systemowym torchlight.
Zastosowano poprawkę main.py.
Zastąpiono przestarzały typ np.int.
Zastąpiono przestarzałe typy NumPy.

Zakończono dostosowanie kodu SkateFormer.


In [ ]:
import yaml, os

WEIGHTS_BY_SPLIT = {
    'CS': '/content/SkateFormer/SkateFormer_j_CSub.pt',
    'CV': '/content/SkateFormer/SkateFormer_j_CView.pt',
}

CONFIGS = [
    ('NTU60_CS.npz',                 'ntu_cs_baseline'),
    ('NTU60_CV.npz',                 'ntu_cv_baseline'),
    ('NTU60_CS_scale.npz',           'ntu_cs_scale'),
    ('NTU60_CV_scale.npz',           'ntu_cv_scale'),
    ('NTU60_CS_both.npz',            'ntu_cs_both'),
    ('NTU60_CV_both.npz',            'ntu_cv_both'),
    ('NTU60_CS_interpolation.npz',   'ntu_cs_interpolation'),
    ('NTU60_CV_interpolation.npz',   'ntu_cv_interpolation'),
    ('NTU60_CS_scale_per_actor.npz', 'ntu_cs_scale_per_actor'),
    ('NTU60_CV_scale_per_actor.npz', 'ntu_cv_scale_per_actor'),
    ('NTU60_CS_both_per_actor.npz',  'ntu_cs_both_per_actor'),
    ('NTU60_CV_both_per_actor.npz',  'ntu_cv_both_per_actor'),
]

for npz_name, run_name in CONFIGS:
    split = 'CS' if npz_name.startswith('NTU60_CS') else 'CV'
    PT_PATH = WEIGHTS_BY_SPLIT[split]

    npz_path = f'./data/ntu/{npz_name}'
    work_dir = f'./work_dir/ntu_sanity/{run_name}/SkateFormer_j'
    cfg_path = f'config/test/ntu_sanity/{run_name}.yaml'

    os.makedirs(work_dir, exist_ok=True)
    os.makedirs(os.path.dirname(cfg_path), exist_ok=True)

    config = {
        'seed': 1,
        'num_worker': 4,
        'work_dir': work_dir,
        'phase': 'test',
        'weights': PT_PATH,
        'feeder': 'feeders.feeder_ntu.Feeder',
        'train_feeder_args': {
            'data_path': npz_path, 'split': 'train', 'debug': False,
            'window_size': 64, 'p_interval': [0.5, 1], 'aug_method': 'a123489',
            'intra_p': 0.5, 'inter_p': 0.2, 'thres': 64, 'uniform': True, 'partition': True
        },
        'test_feeder_args': {
            'data_path': npz_path, 'split': 'test', 'window_size': 64,
            'p_interval': [0.95], 'thres': 64, 'uniform': True, 'partition': True, 'debug': False
        },
        'model': 'model.SkateFormer.SkateFormer_',
        'model_args': {
            'num_classes': 60, 'num_people': 2, 'num_points': 24, 'kernel_size': 7,
            'num_heads': 32, 'attn_drop': 0.5, 'head_drop': 0.0, 'rel': True,
            'drop_path': 0.2, 'type_1_size': [8, 8], 'type_2_size': [8, 12],
            'type_3_size': [8, 8], 'type_4_size': [8, 12], 'mlp_ratio': 4.0, 'index_t': True
        },
        'device': [0], 'batch_size': 128, 'test_batch_size': 128,
        'num_epoch': 1, 'save_score': True
    }

    with open(cfg_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    print(f'\n{"="*60}')
    print(f'Uruchamiam: {run_name}  ({npz_name})  split={split}  weights={PT_PATH.split("/")[-1]}')
    print(f'{"="*60}')
    !python main.py --config {cfg_path}



Uruchamiam: ntu_cs_baseline  (NTU60_CS.npz)  split=CS  weights=SkateFormer_j_CSub.pt
<function SkateFormer_ at 0x7a303a521440>
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
[ Wed Jul 22 20:13:10 2026 ] Load weights from /content/SkateFormer/SkateFormer_j_CSub.pt.
[ Wed Jul 22 20:13:21 2026 ] Model:   model.SkateFormer.SkateFormer_.
[ Wed Jul 22 20:13:21 2026 ] Weights: /content/SkateFormer/SkateFormer_j_CSub.pt.
[ Wed Jul 22 20:13:21 2026 ] Eval epoch: 1
100% 129/129 [00:22<00:00,  5.79it/s]
Accuracy:  0.9262449202401892  model:  
[ Wed Jul 22 20:13:43 2026 ] 	Mean test loss of 129 batches: 0.424409584250561.
[ Wed Jul 22 20:13:44 2026 ] 	Top1: 92.62%
[ Wed Jul 22 20:13:44 2026 ] 	Top5: 98.27%
[ Wed Jul 22 20:13:44 20

In [ ]:
import shutil
shutil.copytree('./work_dir/ntu_sanity', f'{DRIVE_ROOT}/ntu_sanity_results', dirs_exist_ok=True)
print('Wyniki skopiowane na Drive.')

Wyniki skopiowane na Drive.


In [ ]:
import os, pickle, numpy as np

CONFIGS = [
    ('NTU60_CS.npz',                 'ntu_cs_baseline'),
    ('NTU60_CV.npz',                 'ntu_cv_baseline'),
    ('NTU60_CS_scale.npz',           'ntu_cs_scale'),
    ('NTU60_CV_scale.npz',           'ntu_cv_scale'),
    ('NTU60_CS_both.npz',            'ntu_cs_both'),
    ('NTU60_CV_both.npz',            'ntu_cv_both'),
    ('NTU60_CS_interpolation.npz',   'ntu_cs_interpolation'),
    ('NTU60_CV_interpolation.npz',   'ntu_cv_interpolation'),
    ('NTU60_CS_scale_per_actor.npz', 'ntu_cs_scale_per_actor'),
    ('NTU60_CV_scale_per_actor.npz', 'ntu_cv_scale_per_actor'),
    ('NTU60_CS_both_per_actor.npz',  'ntu_cs_both_per_actor'),
    ('NTU60_CV_both_per_actor.npz',  'ntu_cv_both_per_actor'),
]

results = {}
for npz_file, run_name in CONFIGS:
    split = 'CS' if npz_file.startswith('NTU60_CS') else 'CV'
    pkl_path = f'./work_dir/ntu_sanity/{run_name}/SkateFormer_j/epoch1_test_score.pkl'
    if not os.path.exists(pkl_path):
        print(f'BRAK wyników: {run_name}')
        results[run_name] = None
        continue

    with open(pkl_path, 'rb') as f:
        score_dict = pickle.load(f)

    data_path = f'/content/SkateFormer/data/ntu/{npz_file}'
    d = np.load(data_path)
    y_test = np.argmax(d['y_test'], axis=1)

    keys = sorted(score_dict.keys(), key=lambda x: int(x.split('_')[1]))
    scores = np.array([score_dict[k] for k in keys])
    preds = np.argmax(scores, axis=1)
    overall_acc = (preds == y_test).mean() * 100

    interaction_mask = y_test >= 49
    interaction_acc = (preds[interaction_mask] == y_test[interaction_mask]).mean() * 100

    results[run_name] = (overall_acc, interaction_acc)
    print(f'{run_name:24s} split={split}  overall={overall_acc:.2f}%  interakcje={interaction_acc:.2f}%')

print('\n--- TABELA ---')
print(f'{"wariant":16s} {"CS overall":>12s} {"CS interakcje":>15s} {"CV overall":>12s} {"CV interakcje":>15s}')
for mode, cs_run, cv_run in [
    ('baseline',        'ntu_cs_baseline',        'ntu_cv_baseline'),
    ('scale',           'ntu_cs_scale',           'ntu_cv_scale'),
    ('interpolation',   'ntu_cs_interpolation',   'ntu_cv_interpolation'),
    ('both',            'ntu_cs_both',            'ntu_cv_both'),
    ('scale_per_actor', 'ntu_cs_scale_per_actor', 'ntu_cv_scale_per_actor'),
    ('both_per_actor',  'ntu_cs_both_per_actor',  'ntu_cv_both_per_actor'),
]:
    cs = results.get(cs_run)
    cv = results.get(cv_run)
    cs_str = f'{cs[0]:.2f}% / {cs[1]:.2f}%' if cs else 'N/A'
    cv_str = f'{cv[0]:.2f}% / {cv[1]:.2f}%' if cv else 'N/A'
    print(f'{mode:16s} {cs_str:>25s} {cv_str:>25s}')

ntu_cs_baseline          split=CS  overall=92.62%  interakcje=96.83%
ntu_cv_baseline          split=CV  overall=97.02%  interakcje=98.99%
ntu_cs_scale             split=CS  overall=91.31%  interakcje=95.48%
ntu_cv_scale             split=CV  overall=95.98%  interakcje=98.09%
ntu_cs_both              split=CS  overall=91.32%  interakcje=95.54%
ntu_cv_both              split=CV  overall=95.97%  interakcje=98.06%
ntu_cs_interpolation     split=CS  overall=92.61%  interakcje=96.73%
ntu_cv_interpolation     split=CV  overall=97.01%  interakcje=98.93%
ntu_cs_scale_per_actor   split=CS  overall=91.28%  interakcje=95.38%
ntu_cv_scale_per_actor   split=CV  overall=95.88%  interakcje=97.72%
ntu_cs_both_per_actor    split=CS  overall=91.29%  interakcje=95.44%
ntu_cv_both_per_actor    split=CV  overall=95.86%  interakcje=97.63%

--- TABELA ---
wariant            CS overall   CS interakcje   CV overall   CV interakcje
baseline                   92.62% / 96.83%           97.02% / 98.99%
scale       

In [ ]:
#Analiza klas interakcyjnych, wariant NTU CS both
import numpy as np, pickle

RUN    = 'ntu_cs_both'
NPZ    = './data/ntu/NTU60_CS_both.npz'
WDIR   = f'./work_dir/ntu_sanity/{RUN}/SkateFormer_j'

INTERACTION_NTU = list(range(49, 60))
NTU_INT_NAMES = {
    49: 'punch/slap',     50: 'kick person',    51: 'push person',
    52: 'pat on back',    53: 'point at person', 54: 'hugging',
    55: 'giving object',  56: 'touch pocket',    57: 'handshaking',
    58: 'walk toward',    59: 'walk apart'
}

with open(f'{WDIR}/epoch1_test_score.pkl', 'rb') as f:
    score_data = pickle.load(f)

data   = np.load(NPZ)
y_true = np.argmax(data['y_test'], axis=1)
keys   = sorted(score_data.keys(), key=lambda x: int(x.split('_')[1]))
scores = np.array([score_data[k] for k in keys])
y_pred = np.argmax(scores, axis=1)

print(f'NTU CS both — Per-class accuracy (klasy interakcyjne A50-A60)')
print(f'{"NTU Akcja":<22} {"NTU idx":>8} {"Correct":>8} {"Total":>7} {"Acc":>7}')
print('-' * 58)
mask_int = np.isin(y_true, INTERACTION_NTU)
y_ti = y_true[mask_int]
y_pi = y_pred[mask_int]
print(f'Interaction Top1: {(y_ti == y_pi).mean()*100:.2f}%  ({mask_int.sum()} próbek)')
print()
for ntu_idx in INTERACTION_NTU:
    mask    = (y_true == ntu_idx)
    total   = mask.sum()
    if total == 0: continue
    correct = (y_pred[mask] == ntu_idx).sum()
    print(f'{NTU_INT_NAMES[ntu_idx]:<22} {ntu_idx:>8} {correct:>8} {total:>7} {correct/total*100:>6.1f}%')

NTU CS both — Per-class accuracy (klasy interakcyjne A50-A60)
NTU Akcja               NTU idx  Correct   Total     Acc
----------------------------------------------------------
Interaction Top1: 95.54%  (3028 próbek)

punch/slap                   49      259     274   94.5%
kick person                  50      265     276   96.0%
push person                  51      267     276   96.7%
pat on back                  52      236     276   85.5%
point at person              53      263     276   95.3%
hugging                      54      267     274   97.4%
giving object                55      265     276   96.0%
touch pocket                 56      262     275   95.3%
handshaking                  57      268     276   97.1%
walk toward                  58      272     273   99.6%
walk apart                   59      269     276   97.5%


In [ ]:
#Analiza klas interakcyjnych razem
import numpy as np, pickle, os

RUNS = [
    ('ntu_cs_baseline',        './data/ntu/NTU60_CS.npz'),
    ('ntu_cs_scale',           './data/ntu/NTU60_CS_scale.npz'),
    ('ntu_cs_interpolation',   './data/ntu/NTU60_CS_interpolation.npz'),
    ('ntu_cs_both',            './data/ntu/NTU60_CS_both.npz'),
    ('ntu_cs_scale_per_actor', './data/ntu/NTU60_CS_scale_per_actor.npz'),
    ('ntu_cs_both_per_actor',  './data/ntu/NTU60_CS_both_per_actor.npz'),
    ('ntu_cv_baseline',        './data/ntu/NTU60_CV.npz'),
    ('ntu_cv_scale',           './data/ntu/NTU60_CV_scale.npz'),
    ('ntu_cv_interpolation',   './data/ntu/NTU60_CV_interpolation.npz'),
    ('ntu_cv_both',            './data/ntu/NTU60_CV_both.npz'),
    ('ntu_cv_scale_per_actor', './data/ntu/NTU60_CV_scale_per_actor.npz'),
    ('ntu_cv_both_per_actor',  './data/ntu/NTU60_CV_both_per_actor.npz'),
]

INTERACTION_NTU = list(range(49, 60))  #A50-A60, 0-indeksowane 49-59
NTU_INT_NAMES = {
    49: 'punch/slap',      50: 'kick person',     51: 'push person',
    52: 'pat on back',     53: 'point at person',  54: 'hugging',
    55: 'giving object',   56: 'touch pocket',     57: 'handshaking',
    58: 'walk toward',     59: 'walk apart'
}

per_class_results = {}

for run_name, npz_path in RUNS:
    wdir = f'./work_dir/ntu_sanity/{run_name}/SkateFormer_j'
    pkl_path = f'{wdir}/epoch1_test_score.pkl'
    if not os.path.exists(pkl_path):
        print(f'BRAK: {run_name}')
        continue

    with open(pkl_path, 'rb') as f:
        score_data = pickle.load(f)

    data = np.load(npz_path)
    y_true = np.argmax(data['y_test'], axis=1)
    keys = sorted(score_data.keys(), key=lambda x: int(x.split('_')[1]))
    scores = np.array([score_data[k] for k in keys])
    y_pred = np.argmax(scores, axis=1)

    mask_int = np.isin(y_true, INTERACTION_NTU)
    y_ti, y_pi = y_true[mask_int], y_pred[mask_int]
    interaction_top1 = (y_ti == y_pi).mean() * 100

    print(f'\n=== {run_name} — Interaction Top1: {interaction_top1:.2f}% ({mask_int.sum()} próbek) ===')
    print(f'{"NTU Akcja":<22} {"NTU idx":>8} {"Correct":>8} {"Total":>7} {"Acc":>7}')
    print('-' * 58)

    class_accs = {}
    for ntu_idx in INTERACTION_NTU:
        mask = (y_true == ntu_idx)
        total = mask.sum()
        if total == 0:
            continue
        correct = (y_pred[mask] == ntu_idx).sum()
        acc = correct / total * 100
        class_accs[ntu_idx] = acc
        print(f'{NTU_INT_NAMES[ntu_idx]:<22} {ntu_idx:>8} {correct:>8} {total:>7} {acc:>6.1f}%')

    per_class_results[run_name] = {'interaction_top1': interaction_top1, 'per_class': class_accs}


=== ntu_cs_baseline — Interaction Top1: 96.83% (3028 próbek) ===
NTU Akcja               NTU idx  Correct   Total     Acc
----------------------------------------------------------
punch/slap                   49      258     274   94.2%
kick person                  50      266     276   96.4%
push person                  51      271     276   98.2%
pat on back                  52      264     276   95.7%
point at person              53      260     276   94.2%
hugging                      54      273     274   99.6%
giving object                55      264     276   95.7%
touch pocket                 56      264     275   96.0%
handshaking                  57      269     276   97.5%
walk toward                  58      273     273  100.0%
walk apart                   59      270     276   97.8%

=== ntu_cs_scale — Interaction Top1: 95.48% (3028 próbek) ===
NTU Akcja               NTU idx  Correct   Total     Acc
----------------------------------------------------------
punch/slap  